In [27]:
import pandas as pd

In [28]:
df = pd.read_csv(r"C:\Users\91910\Downloads\Brazilian E-Commerce Public Dataset by Olist.csv")

In [29]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [30]:
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%d-%m-%Y %H:%M", errors='coerce')

In [31]:
df.isnull().sum()

order_id                         0
order_item_id                    0
customer_id                      0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
product_id                       0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
seller_id                        0
seller_city                      0
seller_state                     0
seller_zip_code_prefix           0
payment_type                     0
payment_sequential               0
payment_installments             0
price                            0
freight_value                    0
payment_value                    0
shipping_limit_date              0
order_purchase_timestamp         0
order_approved_at   

In [32]:
df = df.dropna(subset=['order_delivered_customer_date'])

In [33]:
df['delivery_delay'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

In [34]:
df['total_order_value'] = df['price'] + df['freight_value']

In [35]:
df['year'] = df['order_purchase_timestamp'].dt.year
df['month'] = df['order_purchase_timestamp'].dt.month

In [36]:
df.to_csv("cleaned_ecommerce.csv", index=False)

In [37]:
import os

In [38]:
os.listdir()

['.ipynb_checkpoints',
 'cleaned_ecommerce.csv',
 'E_Commerce.ipynb',
 'My Music',
 'My Pictures',
 'My Videos']

In [39]:
!pip install pandasql

In [40]:
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [41]:
query = "SELECT * FROM df LIMIT 5"
pysqldf(query)

,order_id,order_item_id,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_id,product_category_name,product_name_lenght,...,day_of_purchase,month_of_purchase,year_of_purchase,month/year_of_purchase,order_status,order_unique_id,delivery_delay,total_order_value,year,month
0,00010242fe8c5a6d1ba2dd792cb16214,1,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58,...,Wednesday,September,2017,Sep-17,delivered,00010242fe8c5a6d1ba2dd792cb16214-1,-9,72.19,2017,9
1,130898c0987d1801452a8ed92a670612,1,e6eecc5a77de221464d1c4eaff0a9b64,0fb8e3eab2d3e79d92bb3fffbb97f188,75800,jatai,GO,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58,...,Wednesday,June,2017,Jun-17,delivered,130898c0987d1801452a8ed92a670612-1,-13,73.86,2017,6
2,532ed5e14e24ae1f0d735b91524b98b9,1,4ef55bf80f711b372afebcb7c715344a,3419052c8c6b45daf79c1e426f9e9bcb,30720,belo horizonte,MG,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58,...,Friday,May,2018,May-18,delivered,532ed5e14e24ae1f0d735b91524b98b9-1,-3,83.23,2018,5
3,6f8c31653edb8c83e1a739408b5ff750,1,30407a72ad8b3f4df4d15369126b20c9,e7c828d22c0682c1565252deefbe334d,83070,sao jose dos pinhais,PR,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58,...,Tuesday,August,2017,Aug-17,delivered,6f8c31653edb8c83e1a739408b5ff750-1,-16,75.07,2017,8
4,7d19f4ef4d04461989632411b7e588b9,1,91a792fef70ecd8cc69d3c7feb3d12da,0bb98ba72dcc08e95f9d8cc434e9a2cc,36400,conselheiro lafaiete,MG,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58,...,Thursday,August,2017,Aug-17,delivered,7d19f4ef4d04461989632411b7e588b9-1,-8,72.19,2017,8


In [42]:
query = """
SELECT COUNT(DISTINCT order_id) AS total_orders
FROM df
"""
pysqldf(query)

,total_orders
0,95128


In [43]:
query = """
SELECT 
SUM(payment_value) / COUNT(DISTINCT order_id) AS AOV
FROM df
"""
pysqldf(query)

,AOV
0,205.307621


In [44]:
query = """
SELECT product_category_name,
SUM(payment_value) AS revenue
FROM df
GROUP BY product_category_name
ORDER BY revenue DESC
LIMIT 10
"""
pysqldf(query)

,product_category_name,revenue
0,cama_mesa_banho,1692557.09
1,beleza_saude,1620868.35
2,informatica_acessorios,1549252.47
3,moveis_decoracao,1393972.04
4,relogios_presentes,1387046.31
5,esporte_lazer,1349194.08
6,utilidades_domesticas,1069787.97
7,automotivo,833610.84
8,ferramentas_jardim,810460.70
9,cool_stuff,744339.94


In [45]:
query = """
SELECT year, month,
SUM(payment_value) AS revenue
FROM df
GROUP BY year, month
ORDER BY year, month
"""
pysqldf(query)

,year,month,revenue
0,2016,10,62495.73
1,2016,12,19.62
2,2017,1,172487.07
3,2017,2,313470.32
4,2017,3,496731.66
5,2017,4,444457.76
6,2017,5,687145.80
7,2017,6,577595.09
8,2017,7,711168.83
9,2017,8,835079.65


In [46]:
query = """
SELECT customer_unique_id,
COUNT(DISTINCT order_id) AS total_orders
FROM df
GROUP BY customer_unique_id
"""
pysqldf(query)

,customer_unique_id,total_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1
...,...,...
92076,fffcf5a5ff07b0908bd4e2dbc735a684,1
92077,fffea47cd6d3cc0a88bd621562a9d061,1
92078,ffff371b4d645b6ecea244b27531430a,1
92079,ffff5962728ec6157033ef9805bacc48,1


In [47]:
query = """
SELECT 
CASE 
    WHEN delivery_delay > 0 THEN 'Late'
    ELSE 'On Time'
END AS delivery_status,
COUNT(*) AS orders
FROM df
GROUP BY delivery_status
"""
pysqldf(query)

,delivery_status,orders
0,Late,7408
1,On Time,105982


In [48]:
df['rank'] = df.groupby('customer_unique_id')['payment_value'] \
               .sum() \
               .rank(ascending=False)

In [49]:
df.columns

Index(['order_id', 'order_item_id', 'customer_id', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_id', 'seller_city', 'seller_state', 'seller_zip_code_prefix',
       'payment_type', 'payment_sequential', 'payment_installments', 'price',
       'freight_value', 'payment_value', 'shipping_limit_date',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'day_of_purchase', 'month_of_purchase',
       'year_of_purchase', 'month/year_of_purchase', 'order_status',
       'order_unique_id', 'delivery_delay', 'total_order_value', 'year',
       'month', 'rank'],
      dtype='object')

In [50]:
df.to_csv("cleaned_ecommerce.csv", index=False)

In [51]:
df.to_csv("cleaned_ecommerce.csv", index=False)

In [53]:
df.to_csv(r"C:\Users\Public\cleaned_ecommerce.csv", index=False)

In [55]:
df.columns

Index(['order_id', 'order_item_id', 'customer_id', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_id', 'seller_city', 'seller_state', 'seller_zip_code_prefix',
       'payment_type', 'payment_sequential', 'payment_installments', 'price',
       'freight_value', 'payment_value', 'shipping_limit_date',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'day_of_purchase', 'month_of_purchase',
       'year_of_purchase', 'month/year_of_purchase', 'order_status',
       'order_unique_id', 'delivery_delay', 'total_order_value', 'year',
       'month', 'rank'],
      dtype='object')